In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Base directory
path = r"dataset/synthetic_dataset/synthetic_mobile_money_transaction_dataset.csv"

df = pd.read_csv(path)
print("Shape : " , df.shape)
df.head()

Shape :  (1720181, 10)


,step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
0,0,TRANSFER,19824.96,4537027967639631,187712.18,167887.22,4875702729424478,8.31,19833.27,1
1,0,PAYMENT,598.97,4296267625767470,8.92,8.92,25-0000401,0.00,0.00,0
2,0,PAYMENT,545.85,4178224023847746,93.60,-452.25,13-0001587,0.00,545.85,0
3,0,TRANSFER,19847.01,4178224023847746,-452.25,-20299.26,4096920916696293,4011.72,23858.74,1
4,0,PAYMENT,546.89,4779013371563747,159148.76,158601.88,75-0003564,0.00,546.89,0


In [3]:
# # Base directory
# path = "dataset/synthetic_dataset/MoMTSim_20240722202413_1000_dataset.csv"

# df = pd.read_csv(path)
# print("Shape : " , df.shape)
# df.head()

In [4]:
df_clean = df.copy()

df_clean.info()
df_clean.describe(include="all").T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1720181 entries, 0 to 1720180
Data columns (total 10 columns):
 #   Column           Dtype  
---  ------           -----  
 0   step             int64  
 1   transactionType  object 
 2   amount           float64
 3   initiator        int64  
 4   oldBalInitiator  float64
 5   newBalInitiator  float64
 6   recipient        object 
 7   oldBalRecipient  float64
 8   newBalRecipient  float64
 9   isFraud          int64  
dtypes: float64(5), int64(3), object(2)
memory usage: 131.2+ MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,1720181.0,NaN,NaN,NaN,65.555286,44.673684,0.0,23.0,54.0,106.0,143.0
transactionType,1720181,5,PAYMENT,667245,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,1720181.0,NaN,NaN,NaN,52538.679612,88356.497983,0.24,606.46,17298.25,71161.49,2142928.28
initiator,1720181.0,NaN,NaN,NaN,4499951976758673.0,289635131225563.0,4000061963235266.0,4248761574606837.0,4508521162599660.0,4750928280020948.0,4999855045426980.0
oldBalInitiator,1720181.0,NaN,NaN,NaN,2433757.597989,1307614.874018,-199997.11,1577186.32,2619827.17,3361338.4,12244691.62
newBalInitiator,1720181.0,NaN,NaN,NaN,2443879.729006,1297180.529754,-199997.11,1600495.69,2625680.49,3361871.54,12244691.62
recipient,1720181,16731,56-0008969,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldBalRecipient,1720181.0,NaN,NaN,NaN,108508.328753,283013.771027,-198368.47,16064.23,63130.18,137382.17,11885539.37
newBalRecipient,1720181.0,NaN,NaN,NaN,122277.215395,319227.67989,-135727.98,24962.33,74481.61,143170.24,12066212.12
isFraud,1720181.0,NaN,NaN,NaN,0.102035,0.302694,0.0,0.0,0.0,0.0,1.0


In [5]:
"transactionType" in df_clean.columns

True

In [6]:
df_clean.drop(columns=["transactionType" , "recipient" , "oldBalRecipient" , "newBalRecipient"], inplace=True, errors='ignore')

df_clean = df_clean[(df_clean["oldBalInitiator"] >= 0) & (df_clean["newBalInitiator"] >= 0)]
df_clean.reset_index(drop=True, inplace=True)

for _col in ['index', 'tx_index', 'Unnamed: 0']:
    if _col in df_clean.columns:
        df_clean.drop(columns=[_col], inplace=True)

df_clean["tx_index"] = np.arange(len(df_clean))

In [7]:
df_clean.head()

,step,amount,initiator,oldBalInitiator,newBalInitiator,isFraud,tx_index
0,0,19824.96,4537027967639631,187712.18,167887.22,1,0
1,0,598.97,4296267625767470,8.92,8.92,0,1
2,0,546.89,4779013371563747,159148.76,158601.88,0,2
3,0,16973.75,4779013371563747,158601.88,141628.12,1,3
4,0,659.02,4479448076398077,261950.48,261291.46,0,4


In [8]:
print(df_clean.dtypes)

step                 int64
amount             float64
initiator            int64
oldBalInitiator    float64
newBalInitiator    float64
isFraud              int64
tx_index             int64
dtype: object


In [9]:
# df_clean = pd.get_dummies(df_clean , columns=["transactionType"] , drop_first=False)
# df_clean.head()

In [10]:
df_clean["amount_balance_ratio"] = df_clean["amount"] / (df_clean["oldBalInitiator"] + 1)
df_clean[["amount" , "oldBalInitiator" , "amount_balance_ratio" , "isFraud"]].head()

,amount,oldBalInitiator,amount_balance_ratio,isFraud
0,19824.96,187712.18,0.105613,1
1,598.97,8.92,60.380040,0
2,546.89,159148.76,0.003436,0
3,16973.75,158601.88,0.107020,1
4,659.02,261950.48,0.002516,0


In [11]:
# df_clean["neg_balance_i"] = (
#     (df_clean["oldBalInitiator"] < 0) &
#     (df_clean["newBalInitiator"] < 0)
# ).astype(int)

# df_clean.head()

In [12]:
df_clean["balance_error_i"] = df_clean["newBalInitiator"] - (df_clean["oldBalInitiator"] - df_clean["amount"])

df_clean["balance_error_i"].head()

0    0.000000e+00
1    5.989700e+02
2    1.000000e-02
3   -1.000000e-02
4   -2.910383e-11
Name: balance_error_i, dtype: float64

In [13]:
# def add_customer_behavior_features(df, windows=[6, 12, 24]):
    
#     df = df.sort_values(["step", "index"]).copy()
    
#     result_list = []

#     for customer_id, group in df.groupby("initiator"):
        
#         group = group.copy()
        
#         for window in windows:
            
#             # Row-based rolling (last N transactions)
#             group[f"INIT_TX_COUNT_{window}"] = (
#                 group["amount"].rolling(window, min_periods=1).count()
#             )
            
#             group[f"INIT_AVG_AMOUNT_{window}"] = (
#                 group["amount"].rolling(window, min_periods=1).mean()
#             )
            
#             group[f"INIT_SUM_AMOUNT_{window}"] = (
#                 group["amount"].rolling(window, min_periods=1).sum()
#             )
        
#         result_list.append(group)

#     df = pd.concat(result_list)
#     df = df.sort_values(["step", "index"]).reset_index(drop=True)
    
#     return df

# df_clean = add_customer_behavior_features(df_clean)

In [14]:
def add_customer_behavior_features(df, tx_windows=[6, 12, 24], step_windows=[6, 12, 24]):
    
    df = df.sort_values(["initiator", "step", "tx_index"]).copy()
    
    result_list = []

    for customer_id, group in df.groupby("initiator"):
        
        group = group.copy()
        
        # Row-based rolling (last N transactions)
        for window in tx_windows:
            
            group[f"INIT_AVG_AMOUNT_TX_{window}"] = (
                group["amount"].rolling(window, min_periods=1).mean()
            )
            
            # Deviation from rolling average
            group[f"INIT_AMOUNT_DEV_TX_{window}"] = (
                group["amount"] - group[f"INIT_AVG_AMOUNT_TX_{window}"]
            )
        
        result_list.append(group)

    df = pd.concat(result_list)
    df = df.sort_values(["step", "tx_index"]).reset_index(drop=True)
    
    return df


df_clean = add_customer_behavior_features(df_clean)
df_clean.head()

,step,amount,initiator,oldBalInitiator,newBalInitiator,isFraud,tx_index,amount_balance_ratio,balance_error_i,INIT_AVG_AMOUNT_TX_6,INIT_AMOUNT_DEV_TX_6,INIT_AVG_AMOUNT_TX_12,INIT_AMOUNT_DEV_TX_12,INIT_AVG_AMOUNT_TX_24,INIT_AMOUNT_DEV_TX_24
0,0,19824.96,4537027967639631,187712.18,167887.22,1,0,0.105613,0.000000e+00,19824.96,0.00,19824.96,0.00,19824.96,0.00
1,0,598.97,4296267625767470,8.92,8.92,0,1,60.380040,5.989700e+02,598.97,0.00,598.97,0.00,598.97,0.00
2,0,546.89,4779013371563747,159148.76,158601.88,0,2,0.003436,1.000000e-02,546.89,0.00,546.89,0.00,546.89,0.00
3,0,16973.75,4779013371563747,158601.88,141628.12,1,3,0.107020,-1.000000e-02,8760.32,8213.43,8760.32,8213.43,8760.32,8213.43
4,0,659.02,4479448076398077,261950.48,261291.46,0,4,0.002516,-2.910383e-11,659.02,0.00,659.02,0.00,659.02,0.00


In [15]:
# def add_recipient_risk_features(df, delay=6, windows=[6, 12, 24]):
    
#     df = df.sort_values(["step", "index"]).copy()
    
#     result_list = []

#     for recipient_id, group in df.groupby("recipient"):
        
#         group = group.copy()
        
#         # shift fraud labels to avoid leakage
#         fraud_shifted = group["isFraud"].shift(delay)
        
#         for window in windows:
            
#             # Rolling over past N transactions
#             fraud_count = fraud_shifted.rolling(window, min_periods=1).sum()
#             tx_count = fraud_shifted.rolling(window, min_periods=1).count()
            
#             # Risk = fraud ratio
#             group[f"RECIP_RISK_{window}"] = fraud_count / (tx_count + 1e-8)
#             group[f"RECIP_TX_COUNT_{window}"] = tx_count
        
#         result_list.append(group)

#     df = pd.concat(result_list)
#     df = df.sort_values(["step", "index"]).reset_index(drop=True)
#     df.fillna(0, inplace=True)
    
#     return df

# df_clean = add_recipient_risk_features(df_clean)

In [16]:
df_clean.head(10)

,step,amount,initiator,oldBalInitiator,newBalInitiator,isFraud,tx_index,amount_balance_ratio,balance_error_i,INIT_AVG_AMOUNT_TX_6,INIT_AMOUNT_DEV_TX_6,INIT_AVG_AMOUNT_TX_12,INIT_AMOUNT_DEV_TX_12,INIT_AVG_AMOUNT_TX_24,INIT_AMOUNT_DEV_TX_24
0,0,19824.96,4537027967639631,187712.18,167887.22,1,0,0.105613,0.000000e+00,19824.960,0.000,19824.960,0.000,19824.960,0.000
1,0,598.97,4296267625767470,8.92,8.92,0,1,60.380040,5.989700e+02,598.970,0.000,598.970,0.000,598.970,0.000
2,0,546.89,4779013371563747,159148.76,158601.88,0,2,0.003436,1.000000e-02,546.890,0.000,546.890,0.000,546.890,0.000
3,0,16973.75,4779013371563747,158601.88,141628.12,1,3,0.107020,-1.000000e-02,8760.320,8213.430,8760.320,8213.430,8760.320,8213.430
4,0,659.02,4479448076398077,261950.48,261291.46,0,4,0.002516,-2.910383e-11,659.020,0.000,659.020,0.000,659.020,0.000
5,0,21103.99,4479448076398077,261291.46,240187.47,1,5,0.080768,0.000000e+00,10881.505,10222.485,10881.505,10222.485,10881.505,10222.485
6,0,648.89,4587942659105635,4808.94,4160.05,0,6,0.134906,9.094947e-13,648.890,0.000,648.890,0.000,648.890,0.000
7,0,16908.07,4587942659105635,4160.05,4160.05,1,7,4.063414,1.690807e+04,8778.480,8129.590,8778.480,8129.590,8778.480,8129.590
8,0,586.53,4978866660478516,78413.40,77826.87,0,8,0.007480,0.000000e+00,586.530,0.000,586.530,0.000,586.530,0.000
9,0,23715.64,4978866660478516,77826.87,54111.24,1,9,0.304719,1.000000e-02,12151.085,11564.555,12151.085,11564.555,12151.085,11564.555


In [17]:
# drop_cols = [
#     "INIT_TX_COUNT_6", 
#     "INIT_TX_COUNT_12", 
#     "INIT_TX_COUNT_24",
#     "transactionType_DEBIT",
#     "transactionType_DEPOSIT",
#     "transactionType_PAYMENT",
#     "transactionType_WITHDRAWAL",
#     "neg_balance",
#     "INIT_AVG_AMOUNT_24",
#     "INIT_SUM_AMOUNT_24"
# ]

# df_clean = df_clean.drop(columns=drop_cols)
# df_clean.head()

In [18]:
save_directory = r"try_08"
os.makedirs(save_directory, exist_ok=True)

df_clean.to_pickle(os.path.join(save_directory , "featured_144days.pkl"))

In [19]:
df_fraud = df_clean[df_clean["isFraud"] == 1].copy()

print("Fraud dataset shape:", df_fraud.shape)
df_fraud.head(10)

Fraud dataset shape: (169348, 15)


,step,amount,initiator,oldBalInitiator,newBalInitiator,isFraud,tx_index,amount_balance_ratio,balance_error_i,INIT_AVG_AMOUNT_TX_6,INIT_AMOUNT_DEV_TX_6,INIT_AVG_AMOUNT_TX_12,INIT_AMOUNT_DEV_TX_12,INIT_AVG_AMOUNT_TX_24,INIT_AMOUNT_DEV_TX_24
0,0,19824.96,4537027967639631,187712.18,167887.22,1,0,0.105613,0.000000e+00,19824.960,0.000,19824.960,0.000,19824.960,0.000
3,0,16973.75,4779013371563747,158601.88,141628.12,1,3,0.107020,-1.000000e-02,8760.320,8213.430,8760.320,8213.430,8760.320,8213.430
5,0,21103.99,4479448076398077,261291.46,240187.47,1,5,0.080768,0.000000e+00,10881.505,10222.485,10881.505,10222.485,10881.505,10222.485
7,0,16908.07,4587942659105635,4160.05,4160.05,1,7,4.063414,1.690807e+04,8778.480,8129.590,8778.480,8129.590,8778.480,8129.590
9,0,23715.64,4978866660478516,77826.87,54111.24,1,9,0.304719,1.000000e-02,12151.085,11564.555,12151.085,11564.555,12151.085,11564.555
16,0,18533.84,4781213214714848,23562.49,5028.65,1,16,0.786549,-1.818989e-12,9556.275,8977.565,9556.275,8977.565,9556.275,8977.565
21,0,21966.03,4120037473988300,318503.12,296537.08,1,21,0.068966,-1.000000e-02,11251.835,10714.195,11251.835,10714.195,11251.835,10714.195
24,0,23981.81,4799013284929379,19240.28,19240.28,1,24,1.246373,2.398181e+04,12289.000,11692.810,12289.000,11692.810,12289.000,11692.810
38,0,17572.83,4836407825782633,19697.16,2124.34,1,38,0.892105,1.000000e-02,9057.655,8515.175,9057.655,8515.175,9057.655,8515.175
41,0,16840.63,4488558934506932,81481.31,64640.69,1,41,0.206678,1.000000e-02,8693.290,8147.340,8693.290,8147.340,8693.290,8147.340


In [20]:
df_clean.columns

Index(['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator',
       'isFraud', 'tx_index', 'amount_balance_ratio', 'balance_error_i',
       'INIT_AVG_AMOUNT_TX_6', 'INIT_AMOUNT_DEV_TX_6', 'INIT_AVG_AMOUNT_TX_12',
       'INIT_AMOUNT_DEV_TX_12', 'INIT_AVG_AMOUNT_TX_24',
       'INIT_AMOUNT_DEV_TX_24'],
      dtype='object')